# Fine-tune `misinfo` and `stance` on a Colab GPU

Both fine-tunes start from a **pre-existing** pretrained checkpoint; neither
trains anything from scratch. This notebook exists only because they want a
GPU. Measured on an 8 GB Apple Silicon laptop over MPS, DeBERTa-v3-base
manages ~2 pairs/sec with gradient checkpointing, which is ~20 h for FNC-1's
50k pairs and ~10 h for misinfo's 27k rows. A T4 does each in well under an
hour.

**Nothing here is new training code.** `modeling/training.py` already holds
the whole path -- load, group-split, baselines, fine-tune, report, error
analysis, registry -- and it is identical on this GPU and on a laptop. This
notebook clones the repo, puts the benchmarks where the loaders expect them,
runs the same CLI you would run locally, and pushes the checkpoint back so
`modeling/registry.py` can pull it.

**Runtime → Change runtime type → T4 GPU** before running anything.


## 1. Check the GPU

If this prints nothing, the runtime is still CPU and everything below will
take the laptop timings rather than the T4 ones.


In [ ]:
!nvidia-smi


## 2. Clone and install

`[modeling]` is the only extra needed: the API, report and source-adapter
trees are irrelevant to training and each pulls a heavy dependency set.


In [ ]:
REPO = 'https://github.com/<you>/narrative-intelligence.git'  # or upload a zip
BRANCH = 'main'

!git clone --depth 1 --branch $BRANCH $REPO /content/ni
%cd /content/ni
!pip -q install -e '.[modeling]'


## 3. Put the benchmarks where the loaders expect them

The loaders resolve `DATA_DIR/benchmarks/<name>/`. FNC-1 is a public clone;
LIAR, FakeNewsNet and CoAID are manual downloads, so the usual move is to
mount Drive and copy whatever you already have on the laptop.

`python -m modeling.cli datasets` prints exactly what is missing and the
steps for each -- run it after this cell and believe it over this comment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# FNC-1 (stance) is open, so it can just be cloned.
!git clone -q https://github.com/FakeNewsChallenge/fnc-1 data/benchmarks/stance

# The misinfo benchmarks are manual downloads. Point this at wherever you
# keep them; the layout under each directory must match what
# `modeling.cli datasets` describes.
BENCH_SRC = '/content/drive/MyDrive/narrative-intelligence/benchmarks'
!mkdir -p data/benchmarks
!cp -rn $BENCH_SRC/liar data/benchmarks/ 2>/dev/null || echo 'no liar copy found'
!cp -rn $BENCH_SRC/fakenewsnet data/benchmarks/ 2>/dev/null || echo 'no fakenewsnet copy found'
!cp -rn $BENCH_SRC/coaid data/benchmarks/ 2>/dev/null || echo 'no coaid copy found'


In [ ]:
!python -m modeling.cli datasets


## 4. Raise the batch size

`configs/models.yaml` is tuned for an 8 GB laptop: batch 16 with gradient
checkpointing on. A T4 has 16 GB and no reason to recompute activations, so
turning checkpointing off and raising the batch is roughly a 2x speedup.

This writes the change into the config rather than passing flags, so the
value that produced the checkpoint is the value recorded in the report.


In [ ]:
import yaml, pathlib

path = pathlib.Path('configs/models.yaml')
config = yaml.safe_load(path.read_text())

config['stance']['batch_size'] = 32
config['stance']['gradient_checkpointing'] = False
config['misinfo']['batch_size'] = 32

path.write_text(yaml.safe_dump(config, sort_keys=False))
print('stance :', {k: config['stance'][k] for k in ('base_model','batch_size','epochs')})
print('misinfo:', {k: config['misinfo'][k] for k in ('base_model','batch_size','epochs')})


> **Note.** Rewriting the YAML drops its comments, and those comments carry
> the reasoning for every value in it. Do not copy this file back to the
> repo -- only the checkpoint under `models/` is meant to come home.


## 5. Train

Same command as on a laptop. Each run saves every epoch, so a dropped Colab
runtime costs one epoch rather than the whole job.

Baselines run *before* the fine-tune, on purpose: the bar is fixed before
anyone has a stake in clearing it. For stance that includes a zero-shot run
of the very same NLI checkpoint, which is the number that actually answers
"was fine-tuning worth it".


In [ ]:
!python -m modeling.cli train stance 2>&1 | tail -40


In [ ]:
!python -m modeling.cli train misinfo 2>&1 | tail -40


## 6. Read the verdict before keeping anything

The project's rule is that a model which does not clear its baselines does
not ship. For misinfo the incumbent to beat is the existing `roberta-base`
run at **macro-F1 0.806**, plus TF-IDF + logistic regression at full scale.
For stance it is the zero-shot NLI baseline.

`clears_every_baseline: false` means the honest outcome is to report it and
keep the incumbent -- not to tune until the number improves.


In [ ]:
import json, pathlib

for module in ('stance', 'misinfo'):
    for metrics_file in sorted(pathlib.Path('artifacts/eval', module).glob('*/metrics.json')):
        payload = json.loads(metrics_file.read_text())
        metrics, baselines = payload['metrics'], payload.get('baselines', {})
        print(f"\n=== {metrics_file.parent} ===")
        print('  is_demo             :', metrics['is_demo'])
        print('  n_test              :', metrics['n_test'])
        print('  macro F1            :', metrics['macro_f1'])
        print('  clears_every_baseline:', baselines.get('clears_every_baseline'))
        for name, verdict in baselines.get('verdicts', {}).items():
            print(f"    vs {name}: {verdict['verdict']} (delta {verdict['delta']:+.4f})")


## 7. Send the checkpoint home

`modeling/registry.py` resolves `models/<module>/<version>/` -> `HF_REPO` ->
`GDRIVE_DIR`, first hit wins, so the layout under whichever you pick must be
exactly `<module>/<version>/`.

Drive is the simpler of the two and needs no token. Use a **private** HF
repo if you choose that route: a checkpoint trained on licensed benchmarks
should not be published by accident.


In [ ]:
# --- option A: Google Drive -----------------------------------------
GDRIVE_DIR = '/content/drive/MyDrive/narrative-intelligence/models'
!mkdir -p $GDRIVE_DIR
!cp -r models/stance  $GDRIVE_DIR/ 2>/dev/null || echo 'no stance checkpoint'
!cp -r models/misinfo $GDRIVE_DIR/ 2>/dev/null || echo 'no misinfo checkpoint'
!find $GDRIVE_DIR -maxdepth 2 -type d


In [ ]:
# --- option B: a private HF model repo ------------------------------
# Set GDRIVE_DIR= and HF_REPO=<user>/<repo> in the laptop's .env to pull.
#
# from huggingface_hub import HfApi, login
# login()  # a write token
# api = HfApi()
# HF_REPO = '<user>/narrative-intelligence-checkpoints'
# api.create_repo(HF_REPO, private=True, exist_ok=True)
# for module in ('stance', 'misinfo'):
#     for version_dir in pathlib.Path('models', module).glob('*'):
#         api.upload_folder(
#             repo_id=HF_REPO,
#             folder_path=str(version_dir),
#             path_in_repo=f'{module}/{version_dir.name}',
#         )


## 8. Bring the evidence back too

The checkpoint is half the deliverable; the evaluation artifacts are the
other half, and they are what the model cards cite. Copy them back and
commit them -- unlike `models/`, `artifacts/` is tracked.


In [ ]:
ART = '/content/drive/MyDrive/narrative-intelligence/artifacts'
!mkdir -p $ART
!cp -r artifacts/eval          $ART/ 2>/dev/null || true
!cp -r artifacts/error_analysis $ART/ 2>/dev/null || true
!find $ART -name 'metrics.json' | sort


## Then, on the laptop

```bash
# .env: GDRIVE_DIR=/path/to/your/synced/Drive/narrative-intelligence/models
#       (or HF_REPO=<user>/<repo> and HF_TOKEN=...)

python -m modeling.cli registry        # confirm the checkpoint resolves
python -m app.cli capabilities         # stance/misinfo should read `ready`
python -m modeling.cli score --all     # score the corpus with it
```

If `capabilities` still says `degraded`, check two things before anything
else: that a real weight file arrived (metadata-only directories are
deliberately refused), and that `registry.json` does not say `is_demo: true`
(fixture-trained checkpoints are refused too).

Finally, update `artifacts/model_cards/stance.md` and `misinfo.md` with the
real numbers, and bump the version in `configs/models.yaml` if the mounted
checkpoint changed -- that version is what every scored row carries in
`model_versions`, and it is how a stale score is detected after a retrain.
